In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import os

from sklearn.model_selection import train_test_split, learning_curve
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from sklearn.datasets import load_breast_cancer #scikit learn alr has the same data set uploaded
from sklearn.metrics import PrecisionRecallDisplay

from xgboost import XGBClassifier

In [ ]:
#LOADING THE DATA IN

data = load_breast_cancer()

#dataframe easier to read
data_frame = pd.DataFrame(data.data, columns = data.feature_names)
#1 = benign, 0 = malignant
data_frame['target'] = data.target

print(f"dataset shape: {data_frame.shape}")
data_frame.head() #outputting polished table,e asier to read

In [ ]:
#PREPROCESSING

#features (x)
x = data_frame.drop('target', axis = 1)
#labels (y)
y = data_frame['target']

#splitting into 80% training and 20% for testing
x_train, x_test, y_train, y_test = train_test_split(x,y, test_size = 0.2, random_state = 42) #need to check random state documentation

#setting mean to 0 and variance to 1 (standardization)
scalar = StandardScaler()
x_train = scalar.fit_transform(x_train)
x_test = scalar.transform(x_test)

In [ ]:
#INITIALIZING MODEL

model = XGBClassifier(
    learning_rate = 0.1,
    max_depth = 3,
    n_estimators = 100,
    use_label_encoder = False,
    eval_metric = 'logloss'
)

In [ ]:
#TRAINING MODEL

#training
model.fit(x_train, y_train)

#debugging
print("training complete")

In [ ]:
#EVALUATION

preds =model.predict(x_test)

print(f"Model accuracy: {accuracy_score(y_test, preds)}")

print("confusion matrix: ")
print(confusion_matrix(y_test, preds))
print("classification: ")
print(classification_report(y_test, preds))

In [ ]:
#VISUALIZATION

plt.figure(figsize= (8,6))
sns.heatmap(confusion_matrix(y_test, preds), annot = True, fmt = 'd', cmap = 'Blues')

plt.xlabel("prediction")
plt.ylabel("actual")
plt.title("xgboost confusion matrix")

plt.show()

if not os.path.exists('figures'):
    os.makedirs('figures')
plt.savefig('figures/xgboost_confusion_matrix.png')


#computing learning curve
train_size, train_score, test_score = learning_curve(model,  x_train, y_train, cv = 5, scoring = 'accuracy', train_sizes = np.linspace(.10, 1.0, 10))

#means and std
train_m = np.mean(train_score, axis = 1)
test_m = np.mean(test_score, axis = 1)

plt.figure(figsize = (10,6))
plt.plot(train_size, train_m, label = 'training accuracy', marker ='o')
plt.plot(train_size, test_m, label = 'cross-val accuracy', marker ='s')
plt.xlabel("training samples (num)")
plt.ylabel("accuracy")
plt.title("xgboost learning curve")

plt.grid(True)
plt.legend()

plt.show()
plt.savefig('figures/xgboost_learning_curve.png')


#precision recall curve
disp = PrecisionRecallDisplay.from_estimator(model, x_test, y_test, name = "xgboost")
plt.show()
plt.savefig('figures/xgboost_precision_recall_curve.png')


In [ ]:
#FEATURE IMPORTANCE

imp = model.feature_importances_
ind = np.argsort(imp)

#plotting
plt.figure(figsize = (12, 8))
plt.title("xgboost feature importance")
plt.bar(range(x.shape[1]), imp[ind], align = "center", color = "seagreen")
plt.xticks(range(x.shape[1]), [data.feature_names[i] for i in ind], rotation = 90)
plt.tight_layout()
plt.show()

plt.savefig('figures/xgboost_feature_importance.png')
